# Urban form and land use
## Mexicali Urban Liveability Index — `WP06_urban_form_and_land_use`

**Lead:** TBC
**Indicators assigned:** 7
**Schema version:** 1.0.0

Land use mix and land use type, residential area, building design and height, dwelling density and urbanisation rate.

Land use type (#405) is the parent covering the permutations formerly listed separately; deliver it once, with the land use classes as measure parameters.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP06_urban_form_and_land_use'
        NOTEBOOK = 'notebooks/06_urban_form_and_land_use.ipynb'

---
## Your indicators (7)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 19 — Building design

`building_design` · *Built Environment · Urban Morphology · Buildings · Building design*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Building morphology and arrangement are critical for housing quality and social interaction, supporting healthy environments through improved urban ventilation.
- **Adapted from:** Chan & Liu, 2018, 'Effects of neighborhood building density, height, greenspace, and cleanliness on indoor environment and health'; Ramponi & Blocken, 2012, 'A computational study on the influence of urban morphology on wind-induced outdoor ventilation'; Yuan & Ng, 2012, 'Building porosity for better urban ventilation'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Design Analysis', 'Airflow simulations']
- **Open questions raised:** I am not sure we can obtain data for this

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_19) at any time to list what is
# still outstanding.
meta_19 = uli.metadata_stub(19, analyst=ANALYST)

# meta_19['rationale']['statement'] = """..."""
# meta_19['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_19['rationale']['arid_context'] = '...'
# meta_19['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_19['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_19)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_19 = 'grid_100m'
METHOD_19 = 'population_weighted_mean'

native_19 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_19 = uli.harmonise(
    native_19,
    native_scale=NATIVE_SCALE_19,
    method=METHOD_19,
)
results_19 = uli.label(
    harmonised_19,
    meta_19,
    measure_id='building_design__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_19, meta_19))
# uli.write_indicator(results_19, meta_19)

### 22 — Building height

`building_height` · *Built Environment · Urban Morphology · Buildings · Building height*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Variation in building heights within a neighbourhood improves natural ventilation and passive pollutant dispersion while supporting residential privacy and safety.
- **Adapted from:** Ng, 2010, 'Designing high density cities for social & environmental sustainability'; An et al., 2019, 'Exploration of sustainable building morphologies for effective passive pollutant dispersion'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Design Analysis', 'Airflow simulations']
- **Open questions raised:** ER: Can we calculate this with OpenBuildings?

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_22) at any time to list what is
# still outstanding.
meta_22 = uli.metadata_stub(22, analyst=ANALYST)

# meta_22['rationale']['statement'] = """..."""
# meta_22['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_22['rationale']['arid_context'] = '...'
# meta_22['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_22['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_22)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_22 = 'grid_100m'
METHOD_22 = 'population_weighted_mean'

native_22 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_22 = uli.harmonise(
    native_22,
    native_scale=NATIVE_SCALE_22,
    method=METHOD_22,
)
results_22 = uli.label(
    harmonised_22,
    meta_22,
    measure_id='building_height__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_22, meta_22))
# uli.write_indicator(results_22, meta_22)

### 338 — Dwelling density

`dwelling_density` · *Sociodemographics · Household · Demographics · Dwelling density*

- **Lenses to deliver:** density
- **Draft rationale (rewrite this):** Appropriate household density within dwelling units is key to preventing the negative social and health consequences of urban overcrowding.
- **Adapted from:** Reynolds-Salmon et al., 2024, 'Does household size matter? Crowding and its effects on child development'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** 16: ['Exploratory Factor Analysis', 'Standard Deviational Ellipse,', 'Hot Spot Analysis,', 'Network Analysis'] 27: indicating the number of housing units per acre
- **Candidate data sources:** INEGI Census

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_338) at any time to list what is
# still outstanding.
meta_338 = uli.metadata_stub(338, analyst=ANALYST)

# meta_338['rationale']['statement'] = """..."""
# meta_338['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_338['rationale']['arid_context'] = '...'
# meta_338['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_338['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_338)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_338 = 'grid_100m'
METHOD_338 = 'population_weighted_mean'

native_338 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_338 = uli.harmonise(
    native_338,
    native_scale=NATIVE_SCALE_338,
    method=METHOD_338,
)
results_338 = uli.label(
    harmonised_338,
    meta_338,
    measure_id='dwelling_density__density',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_338, meta_338))
# uli.write_indicator(results_338, meta_338)

### 405 — Land use type

`land_use_type` · *Ambient Environment · Resources · Land · Land use type*

- **Lenses to deliver:** quality
- **Candidate data sources:** https://www.mexicali.gob.mx/sitioimip/geovisor/geovisor/?url=pducpmp&access_token= IMIP: usos de suelo https://tecmx.sharepoint.com/:f:/s/Mexicali-SIUM/IgCeSV3JIwQpT6TXm52Vw2gqAXN8KTu7Dex3TtPfbb754oo?e=4QOX5y
- **Open questions raised:** ER: Could be used to calculate land use mix

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_405) at any time to list what is
# still outstanding.
meta_405 = uli.metadata_stub(405, analyst=ANALYST)

# meta_405['rationale']['statement'] = """..."""
# meta_405['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_405['rationale']['arid_context'] = '...'
# meta_405['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_405['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_405)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_405 = 'grid_100m'
METHOD_405 = 'population_weighted_mean'

native_405 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_405 = uli.harmonise(
    native_405,
    native_scale=NATIVE_SCALE_405,
    method=METHOD_405,
)
results_405 = uli.label(
    harmonised_405,
    meta_405,
    measure_id='land_use_type__quality',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_405, meta_405))
# uli.write_indicator(results_405, meta_405)

### 116 — Land use mix

`land_use_mix` · *Built Environment · Urban Morphology · Land Use · Land use mix*

- **Lenses to deliver:** quality
- **Draft rationale (rewrite this):** Source article not available for review.
- **Adapted from:** Not reported.
- **Effect reported there:** Not reported.
- **Methods used in the literature:** ['GIS-based spatial diagnostics']
- **Candidate data sources:** https://www.mexicali.gob.mx/sitioimip/geovisor/geovisor/?url=pducpmp&access_token= IMIP: usos de suelo https://tecmx.sharepoint.com/:f:/s/Mexicali-SIUM/IgCeSV3JIwQpT6TXm52Vw2gqAXN8KTu7Dex3TtPfbb754oo?e=4QOX5y

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_116) at any time to list what is
# still outstanding.
meta_116 = uli.metadata_stub(116, analyst=ANALYST)

# meta_116['rationale']['statement'] = """..."""
# meta_116['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_116['rationale']['arid_context'] = '...'
# meta_116['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_116['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_116)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_116 = 'grid_100m'
METHOD_116 = 'population_weighted_mean'

native_116 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_116 = uli.harmonise(
    native_116,
    native_scale=NATIVE_SCALE_116,
    method=METHOD_116,
)
results_116 = uli.label(
    harmonised_116,
    meta_116,
    measure_id='land_use_mix__quality',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_116, meta_116))
# uli.write_indicator(results_116, meta_116)

### 230 — Residential area

`residential_area` · *Built Environment · Urban Morphology · Land Use · Residential area*

- **Lenses to deliver:** density
- **Draft rationale (rewrite this):** Adequate residential area per capita is important for ensuring privacy, comfort, and a high standard of living within urban dwellings.
- **Adapted from:** Saeed et al., 2022, 'An integrated approach for developing an urban livability composite index—A cities’ ranking road map to achieve urban sustainability'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Exploratory Factor Analysis', 'Standard Deviational Ellipse,', 'Hot Spot Analysis,', 'Network Analysis']
- **Candidate data sources:** https://www.mexicali.gob.mx/sitioimip/geovisor/geovisor/?url=pducpmp&access_token= IMIP: usos de suelo https://tecmx.sharepoint.com/:f:/s/Mexicali-SIUM/IgCeSV3JIwQpT6TXm52Vw2gqAXN8KTu7Dex3TtPfbb754oo?e=4QOX5y
- **Open questions raised:** ER: Could be used to calculate land use mix

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_230) at any time to list what is
# still outstanding.
meta_230 = uli.metadata_stub(230, analyst=ANALYST)

# meta_230['rationale']['statement'] = """..."""
# meta_230['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_230['rationale']['arid_context'] = '...'
# meta_230['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_230['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_230)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_230 = 'grid_100m'
METHOD_230 = 'population_weighted_mean'

native_230 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_230 = uli.harmonise(
    native_230,
    native_scale=NATIVE_SCALE_230,
    method=METHOD_230,
)
results_230 = uli.label(
    harmonised_230,
    meta_230,
    measure_id='residential_area__density',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_230, meta_230))
# uli.write_indicator(results_230, meta_230)

### 335 — Urbanization rate

`urbanization_rate` · *Sociodemographics · City · Urbanization rate*

- **Lenses to deliver:** none flagged in the workbook
- **Draft rationale (rewrite this):** Rapid urbanization can place excessive strain on the fragile ecosystems of underdeveloped regions, potentially diminishing the quality of the living environment.
- **Adapted from:** Fu et al., 2017, 'Hydrogeomorphic ecosystem responses to natural and anthropogenic changes in the Loess Plateau of China'; Bediroglu, 2021, 'Developing GIS interface for the automated analysis and assessment of livable and sustainable city criterion sets'
- **Effect reported there:** Urban liveability: β = -0.082 (p < 0.05), inverse association
- **Methods used in the literature:** ['Entropy TOPSIS model', 'Tobit model']
- **Candidate data sources:** GHSL
- **Team notes:** ER: This can be measured with the longitudinal data and analysis we have on our scaling and remoteness article

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_335) at any time to list what is
# still outstanding.
meta_335 = uli.metadata_stub(335, analyst=ANALYST)

# meta_335['rationale']['statement'] = """..."""
# meta_335['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_335['rationale']['arid_context'] = '...'
# meta_335['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_335['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_335)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_335 = 'grid_100m'
METHOD_335 = 'population_weighted_mean'

native_335 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_335 = uli.harmonise(
    native_335,
    native_scale=NATIVE_SCALE_335,
    method=METHOD_335,
)
results_335 = uli.label(
    harmonised_335,
    meta_335,
    measure_id='urbanization_rate__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_335, meta_335))
# uli.write_indicator(results_335, meta_335)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()